# Silver to Gold - Northstar Transactions

This notebook reads `silver.transactions_clean`, builds a simple dimensional model, and writes Gold Lakehouse tables for analytics.

Before running:
1. Attach the `silver` and `gold` Lakehouses to this notebook.
2. Run `BronzeToSilver.ipynb` successfully first.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## Configure Lakehouse names


In [ ]:
silver_lakehouse = "silver"
gold_lakehouse = "gold"
silver_table = "transactions_clean"


## Read clean silver data


In [ ]:
tx = spark.table(f"{silver_lakehouse}.{silver_table}")

silver_count = tx.count()
print(f"Silver rows: {silver_count:,}")
display(tx.limit(10))


## Build dimension tables


In [ ]:
dim_date = (
    tx.select("TransactionDate", "TransactionYear", "TransactionMonth", "TransactionDay")
      .where(F.col("TransactionDate").isNotNull())
      .dropDuplicates(["TransactionDate"])
      .withColumn("DateKey", F.date_format("TransactionDate", "yyyyMMdd").cast("int"))
      .select("DateKey", "TransactionDate", "TransactionYear", "TransactionMonth", "TransactionDay")
)

dim_region = (
    tx.select("Region", "Country")
      .fillna({"Region": "Unknown", "Country": "Unknown"})
      .dropDuplicates()
      .withColumn("RegionKey", F.dense_rank().over(Window.orderBy("Region", "Country")))
      .select("RegionKey", "Region", "Country")
)

dim_merchant_category = (
    tx.select("MerchantCategory")
      .fillna({"MerchantCategory": "Unknown"})
      .dropDuplicates()
      .withColumn("MerchantCategoryKey", F.dense_rank().over(Window.orderBy("MerchantCategory")))
      .select("MerchantCategoryKey", "MerchantCategory")
)

dim_auth_status = (
    tx.select("AuthStatus")
      .fillna({"AuthStatus": "UNKNOWN"})
      .dropDuplicates()
      .withColumn("AuthStatusKey", F.dense_rank().over(Window.orderBy("AuthStatus")))
      .select("AuthStatusKey", "AuthStatus")
)

display(dim_date.limit(10))
display(dim_region)
display(dim_merchant_category)
display(dim_auth_status)


## Build fact table


In [ ]:
t = tx.alias("t")
r = dim_region.alias("r")
m = dim_merchant_category.alias("m")
a = dim_auth_status.alias("a")

fact_transactions = (
    t.withColumn("DateKey", F.date_format("TransactionDate", "yyyyMMdd").cast("int"))
     .join(r, (F.coalesce(F.col("t.Region"), F.lit("Unknown")) == F.col("r.Region")) & (F.coalesce(F.col("t.Country"), F.lit("Unknown")) == F.col("r.Country")), "left")
     .join(m, F.coalesce(F.col("t.MerchantCategory"), F.lit("Unknown")) == F.col("m.MerchantCategory"), "left")
     .join(a, F.coalesce(F.col("t.AuthStatus"), F.lit("UNKNOWN")) == F.col("a.AuthStatus"), "left")
     .select(
         F.col("t.TransactionID"),
         F.col("DateKey"),
         F.col("r.RegionKey"),
         F.col("m.MerchantCategoryKey"),
         F.col("a.AuthStatusKey"),
         F.col("t.MerchantID"),
         F.col("t.CardToken"),
         F.col("t.Amount"),
         F.col("t.Currency"),
         F.col("t.ProcessingFee"),
         F.col("t.IsInternational"),
         F.col("t.HighValueFlag"),
         F.col("t.AmountBand"),
         F.col("t.Timestamp")
     )
)

display(fact_transactions.limit(10))


## Write Gold Delta tables


In [ ]:
tables = {
    "dim_date": dim_date,
    "dim_region": dim_region,
    "dim_merchant_category": dim_merchant_category,
    "dim_auth_status": dim_auth_status,
    "fact_transactions": fact_transactions
}

for table_name, df in tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{gold_lakehouse}.{table_name}")
    )
    print(f"Wrote {gold_lakehouse}.{table_name}: {df.count():,} rows")


## Validate Gold model


In [ ]:
display(spark.sql(f"""
SELECT
    r.Region,
    m.MerchantCategory,
    a.AuthStatus,
    COUNT(*) AS TransactionCount,
    SUM(f.Amount) AS TotalAmount,
    AVG(f.Amount) AS AverageAmount
FROM {gold_lakehouse}.fact_transactions f
LEFT JOIN {gold_lakehouse}.dim_region r ON f.RegionKey = r.RegionKey
LEFT JOIN {gold_lakehouse}.dim_merchant_category m ON f.MerchantCategoryKey = m.MerchantCategoryKey
LEFT JOIN {gold_lakehouse}.dim_auth_status a ON f.AuthStatusKey = a.AuthStatusKey
GROUP BY r.Region, m.MerchantCategory, a.AuthStatus
ORDER BY TotalAmount DESC
LIMIT 25
"""))
